## 라이브러리 호출

In [1]:
# 라이브러리 호출
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
import platform
import ast
from collections import Counter
import json

warnings.filterwarnings('ignore')

# 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # Mac
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 컬럼 너비 제한 해제
pd.set_option('display.max_colwidth', None)

---
## 데이터 로드

In [ ]:
# 원본 데이터 로드
df_platinum_Match = pd.read_csv('./tft_game_dataset/TFT_Platinum_MatchData.csv')
df_diamond_Match = pd.read_csv('./tft_game_dataset/TFT_Diamond_MatchData.csv')
df_master_Match = pd.read_csv('./tft_game_dataset/TFT_Master_MatchData.csv')
df_grand_master_Match = pd.read_csv('./tft_game_dataset/TFT_GrandMaster_MatchData.csv')
df_challenger_Match = pd.read_csv('./tft_game_dataset/TFT_Challenger_MatchData.csv')

df_Champion_info = pd.read_csv('./tft_game_dataset/TFT_Champion_CurrentVersion.csv')
df_Items_info = pd.read_csv('./tft_game_dataset/TFT_Item_CurrentVersion.csv')


# 매치관련 데이터가 담긴 딕셔너리 정의
match_data = {
    'platinum': df_platinum_Match,
    'diamond': df_diamond_Match,
    'master': df_master_Match,
    'grand_master': df_grand_master_Match,
    'challenger': df_challenger_Match,
}

---
## Match 관련 데이터 병합 (행 기준)

In [ ]:
# 각 티어별 테이블에 'Tier'정보가 담긴 컬럼 추가
for name, df in match_data.items():
    df['tier'] = name

# 티어가 잘 추가되었는지 확인하는 용도
# for name, df in match_data.items():
#     print(f"=== {name} ===")
#     print(df['tier'].value_counts())
#     print()

=== Platinum ===
tier
Platinum    80000
Name: count, dtype: int64

=== Diamond ===
tier
Diamond    80000
Name: count, dtype: int64

=== Master ===
tier
Master    79999
Name: count, dtype: int64

=== GrandMaster ===
tier
GrandMaster    80000
Name: count, dtype: int64

=== Challenger ===
tier
Challenger    79999
Name: count, dtype: int64



In [8]:
# 모든 티어의 경기데이터가 담긴 데이터프레임 제작
# ignore_index=True: 데이터를 병합한 뒤, 새로운 인덱스 부여
df_all_match = pd.concat(match_data.values(), ignore_index=True)

df_all_match.head(2)

,gameId,gameDuration,level,lastRound,Ranked,ingameDuration,combination,champion,tier
0,KR_4291707834,1963.905273,6,27,5,1390.165771,"{'Cybernetic': 1, 'Demolitionist': 1, 'Infiltrator': 1, 'Rebel': 1, 'Set3_Brawler': 1, 'Set3_Celestial': 1, 'Set3_Void': 1, 'Sniper': 1}","{'Ziggs': {'items': [7], 'star': 1}, 'Ashe': {'items': [9], 'star': 1}, 'ChoGath': {'items': [6], 'star': 1}, 'Ekko': {'items': [1], 'star': 1}}",Platinum
1,KR_4291707834,1963.905273,8,37,3,1891.282715,"{'Blaster': 1, 'Chrono': 1, 'Cybernetic': 4, 'Demolitionist': 1, 'Rebel': 1, 'Set3_Blademaster': 2, 'Set3_Brawler': 1, 'Set3_Sorcerer': 1, 'Set3_Void': 1, 'Valkyrie': 1, 'Vanguard': 2}","{'Ziggs': {'items': [24], 'star': 3}, 'Fiora': {'items': [37], 'star': 2}, 'Leona': {'items': [36, 24], 'star': 2}, 'Lucian': {'items': [], 'star': 2}, 'Vi': {'items': [5], 'star': 2}, 'Kayle': {'items': [], 'star': 2}, 'WuKong': {'items': [3, 67], 'star': 2}, 'VelKoz': {'items': [4], 'star': 2}}",Platinum


In [20]:
len(df_all_match[df_all_match['Ranked']==0]['gameId'])

92

In [21]:
print(f"'ranked==0' 행 수: {len(df_all_match[df_all_match['Ranked']==0])}")
print(f"전체 행 수: {len(df_all_match)}")
print(f"비율: {round(len(df_all_match[df_all_match['Ranked']==0]) / len(df_all_match) * 100, 3)}%")

'ranked==0' 행 수: 92
전체 행 수: 399998
비율: 0.023%


In [22]:
df_all_match[df_all_match['Ranked']==0]['gameId'].unique()

<StringArray>
['KR_4229542485', 'KR_4327058667', 'KR_4327053826', 'KR_4236285365',
 'KR_3890408252', 'KR_4231144753', 'KR_4323594767', 'KR_3891808329',
 'KR_3891442705', 'KR_3891371111', 'KR_4231208133', 'KR_4295022760',
 'KR_4303195386', 'KR_4336328756']
Length: 14, dtype: str

In [23]:
# 게임 아이디별로 ranked == 0이라서 지워지는 행 수
# size() : 그룹별 행 수
df_all_match[df_all_match['Ranked']==0].groupby('gameId').size()

gameId
KR_3890408252    8
KR_3891371111    8
KR_3891442705    8
KR_3891808329    8
KR_4229542485    7
KR_4231144753    7
KR_4231208133    7
KR_4236285365    7
KR_4295022760    7
KR_4303195386    7
KR_4323594767    7
KR_4327053826    2
KR_4327058667    2
KR_4336328756    7
dtype: int64

In [ ]:
df_all_match[df_all_match['gameId'].isin(['KR_4327053826'])] # 1,2 데이터 손실

,gameId,gameDuration,level,lastRound,Ranked,ingameDuration,combination,champion,tier
15648,KR_4327053826,1210.759399,5,13,8,601.587585,"{'ManaReaver': 1, 'SpacePirate': 1}","{'Darius': {'items': [17, 23], 'star': 2}}",Platinum
15649,KR_4327053826,1210.759399,9,25,3,1180.209961,{},{},Platinum
15650,KR_4327053826,1210.759399,6,26,0,1210.759399,"{'DarkStar': 1, 'Demolitionist': 1, 'Infiltrator': 2, 'Mercenary': 1, 'Set3_Brawler': 1, 'Set3_Void': 2, 'SpacePirate': 1}","{'KhaZix': {'items': [1], 'star': 1}, 'Shaco': {'items': [3], 'star': 1}, 'ChoGath': {'items': [7], 'star': 1}, 'Gangplank': {'items': [2], 'star': 1}}",Platinum
15651,KR_4327053826,1210.759399,5,13,7,639.365356,"{'Blaster': 1, 'Chrono': 1, 'Cybernetic': 1, 'DarkStar': 1, 'ManaReaver': 1, 'Protector': 1, 'Set3_Celestial': 1, 'Set3_Sorcerer': 1, 'StarGuardian': 1, 'Vanguard': 1}","{'TwistedFate': {'items': [], 'star': 1}, 'Poppy': {'items': [], 'star': 1}, 'JarvanIV': {'items': [6], 'star': 1}, 'Lucian': {'items': [], 'star': 1}, 'Kassadin': {'items': [7], 'star': 1}}",Platinum
15652,KR_4327053826,1210.759399,5,20,5,942.642700,"{'Blaster': 1, 'Chrono': 1, 'Rebel': 1, 'Set3_Blademaster': 1, 'Set3_Brawler': 1, 'Set3_Celestial': 1}","{'Xayah': {'items': [7], 'star': 1}, 'Blitzcrank': {'items': [2], 'star': 1}, 'Jinx': {'items': [6], 'star': 1}}",Platinum
15653,KR_4327053826,1210.759399,5,14,6,669.454773,"{'Blaster': 4, 'Chrono': 1, 'Cybernetic': 1, 'Demolitionist': 1, 'Rebel': 3, 'Set3_Blademaster': 1, 'SpacePirate': 1}","{'Ziggs': {'items': [], 'star': 2}, 'Graves': {'items': [35], 'star': 2}, 'Lucian': {'items': [44], 'star': 2}, 'Ezreal': {'items': [], 'star': 1}, 'MasterYi': {'items': [], 'star': 1}, 'Jinx': {'items': [], 'star': 1}}",Platinum
15654,KR_4327053826,1210.759399,5,21,4,1027.783691,"{'Blaster': 1, 'Cybernetic': 1, 'Demolitionist': 1, 'MechPilot': 1}","{'Lucian': {'items': [6], 'star': 1}, 'Rumble': {'items': [4], 'star': 1}}",Platinum
15655,KR_4327053826,1210.759399,7,26,0,1210.759399,"{'Chrono': 1, 'DarkStar': 2, 'Infiltrator': 1, 'Protector': 2, 'Rebel': 1, 'Set3_Celestial': 3, 'Set3_Mystic': 2, 'Sniper': 3, 'StarGuardian': 1}","{'Caitlyn': {'items': [], 'star': 2}, 'Sona': {'items': [], 'star': 2}, 'Rakan': {'items': [], 'star': 2}, 'XinZhao': {'items': [], 'star': 1}, 'Shaco': {'items': [], 'star': 2}, 'Ashe': {'items': [], 'star': 2}, 'Soraka': {'items': [], 'star': 1}, 'Jhin': {'items': [19, 26], 'star': 1}}",Platinum


In [ ]:
df_all_match[df_all_match['gameId'].isin(['KR_4327058667'])] # 1,2등 데이터 손실

,gameId,gameDuration,level,lastRound,Ranked,ingameDuration,combination,champion,tier
15632,KR_4327058667,931.473633,5,18,4,915.463013,"{'Demolitionist': 1, 'Rebel': 1, 'Set3_Mystic': 1, 'SpacePirate': 1, 'StarGuardian': 1, 'Vanguard': 1}","{'Ziggs': {'items': [4], 'star': 1}, 'Jayce': {'items': [4], 'star': 1}, 'Soraka': {'items': [3], 'star': 1}}",Platinum
15633,KR_4327058667,931.473633,5,13,8,653.917297,"{'Blaster': 1, 'DarkStar': 1, 'Infiltrator': 1, 'Set3_Sorcerer': 1, 'SpacePirate': 1, 'StarGuardian': 2, 'Vanguard': 1}","{'Graves': {'items': [], 'star': 1}, 'Poppy': {'items': [], 'star': 1}, 'Zoe': {'items': [3], 'star': 1}, 'Shaco': {'items': [3], 'star': 1}}",Platinum
15634,KR_4327058667,931.473633,6,14,7,709.650330,"{'DarkStar': 1, 'Protector': 1, 'Set3_Celestial': 1, 'Sniper': 1}","{'JarvanIV': {'items': [2], 'star': 1}, 'Ashe': {'items': [1], 'star': 1}}",Platinum
15635,KR_4327058667,931.473633,6,16,6,799.693726,"{'Blaster': 1, 'Chrono': 3, 'Infiltrator': 1, 'Rebel': 1, 'Set3_Blademaster': 2, 'Set3_Brawler': 1, 'Set3_Celestial': 1, 'Set3_Sorcerer': 1, 'StarGuardian': 1, 'Valkyrie': 1, 'Vanguard': 1}","{'TwistedFate': {'items': [], 'star': 1}, 'Malphite': {'items': [1], 'star': 1}, 'Poppy': {'items': [], 'star': 1}, 'Xayah': {'items': [], 'star': 1}, 'KaiSa': {'items': [], 'star': 1}, 'Shen': {'items': [], 'star': 1}, 'Ezreal': {'items': [7], 'star': 1}}",Platinum
15636,KR_4327058667,931.473633,5,18,3,929.300537,"{'Chrono': 1, 'DarkStar': 1, 'Infiltrator': 1, 'Protector': 1, 'Rebel': 1, 'Set3_Blademaster': 1, 'Set3_Brawler': 1, 'Set3_Celestial': 1, 'Sniper': 1, 'StarGuardian': 1, 'Valkyrie': 1, 'Vanguard': 1}","{'Malphite': {'items': [], 'star': 1}, 'Poppy': {'items': [6], 'star': 2}, 'KaiSa': {'items': [], 'star': 1}, 'Shen': {'items': [], 'star': 1}, 'XinZhao': {'items': [2], 'star': 1}, 'Jhin': {'items': [3], 'star': 1}}",Platinum
15637,KR_4327058667,931.473633,5,18,0,931.473633,"{'Infiltrator': 1, 'MechPilot': 1, 'Set3_Sorcerer': 2, 'Set3_Void': 1, 'StarGuardian': 1}","{'KhaZix': {'items': [9], 'star': 1}, 'Annie': {'items': [6], 'star': 1}, 'Syndra': {'items': [1], 'star': 1}}",Platinum
15638,KR_4327058667,931.473633,6,16,5,847.925537,"{'Chrono': 1, 'Cybernetic': 1, 'Protector': 1, 'Rebel': 2, 'Set3_Blademaster': 2, 'Set3_Brawler': 1, 'Set3_Celestial': 2, 'Set3_Mystic': 1, 'Sniper': 1}","{'Malphite': {'items': [], 'star': 1}, 'Fiora': {'items': [5], 'star': 1}, 'Caitlyn': {'items': [], 'star': 1}, 'Xayah': {'items': [], 'star': 1}, 'Sona': {'items': [5], 'star': 1}, 'XinZhao': {'items': [], 'star': 1}}",Platinum
15639,KR_4327058667,931.473633,5,18,0,931.473633,"{'Demolitionist': 1, 'MechPilot': 1, 'Protector': 1, 'Set3_Blademaster': 1, 'Set3_Celestial': 2}","{'Xayah': {'items': [7], 'star': 1}, 'Rakan': {'items': [9], 'star': 1}, 'Rumble': {'items': [4], 'star': 1}}",Platinum
